# NER truyện tiên hiệp: huấn luyện và kiểm tra trọn gói trên Google Colab

Chỉ cần **Runtime → Run all**, notebook sẽ làm lần lượt:

1. Kiểm tra GPU, cài thư viện
2. Kết nối Google Drive, lấy bộ công cụ
3. Chuẩn bị dữ liệu (4 lựa chọn, xem Bước 3)
4. Huấn luyện ViDeBERTa (`Fsoft-AIC/videberta-base`), đánh giá F1 từng nhãn trên tập test, phân tích lỗi
5. **Lưu mô hình và báo cáo lên Drive**, để không mất khi hết phiên Colab
6. Chạy thử mô hình: câu mẫu, cả thư mục truyện (CSV và HTML tô màu), gộp tên đồng nhất
7. Chạy thử API
8. Đóng gói mô hình để tải về

## Trước khi chạy

1. Trong **My Drive**, tạo thư mục `ner_tien_hiep`.
2. Tải lên thư mục đó file **`bo_cong_cu_ner_tien_hiep_v1.3.zip`** (hoặc bản mới hơn) và file notebook này.
3. Mở notebook: trong Drive, nhấp đúp file `.ipynb`, chọn **Open with → Google Colaboratory**.
4. Chọn **Runtime → Change runtime type → T4 GPU → Save**.
5. Chọn **Runtime → Run all**. Khi Colab hỏi quyền truy cập Drive, chọn tài khoản và cho phép.

**Lần đầu, nên giữ nguyên cấu hình** (`DATA_SOURCE = demo`) để chạy thử toàn bộ quy trình. Ước tính khoảng 10–20 phút trên T4. Con số này chưa đo, thời gian thật tuỳ GPU Colab cấp.

## Thư mục trên Drive

```
MyDrive/ner_tien_hiep/
├── bo_cong_cu_ner_tien_hiep_v1.3.zip   ← bạn tải lên
├── exports/        (tuỳ chọn) file export Label Studio .json, dùng khi DATA_SOURCE = label_studio
├── data/           (tuỳ chọn) train.jsonl, dev.jsonl, test.jsonl, dùng khi DATA_SOURCE = jsonl
├── truyen/         (tuỳ chọn) truyện .txt để chạy thử mô hình, mỗi bộ truyện 1 thư mục con
├── tu_dien.json    (tuỳ chọn) từ điển tên, bổ sung tên mô hình bỏ sót
├── bi_danh.json    (tự tạo) khi áp dụng kết quả duyệt gộp tên ở Bước 10b
├── models/ner-final-<thời gian>/       ← MÔ HÌNH ĐÃ HUẤN LUYỆN
├── checkpoints/<thời gian>/            ← (nếu bật) để chạy tiếp khi mất kết nối
└── runs/<thời gian>/                   ← kết quả mỗi lần chạy
    ├── bao_cao/   (F1 từng nhãn, câu đoán sai, model_info.json)
    ├── ket_qua/   (thực thể tìm được trong truyện: CSV, HTML)
    ├── lien_ket/  (gộp tên: thuc_the.csv, can_duyet.csv…)
    └── data/      (bản sao dữ liệu đã dùng để huấn luyện)
```

## Chỉ muốn kiểm tra một mô hình đã có (không huấn luyện lại)

1. Chạy Bước 1 đến Bước 4.
2. Ở Bước 8, điền `MODEL_FOR_TEST` là thư mục mô hình trên Drive.
3. Chạy lần lượt từng ô từ Bước 8 trở đi bằng **Shift + Enter**.

> ⚠️ **Dữ liệu:** đưa truyện có bản quyền hoặc dữ liệu nội bộ lên Google Drive/Colab tức là đưa dữ liệu ra ngoài hạ tầng của tổ chức. Hãy kiểm tra quy định nội bộ trước. Dữ liệu mẫu trong bộ công cụ là giả lập nên dùng thoải mái.
>
> Notebook cũng chạy được trên Jupyter khác, ví dụ Amazon SageMaker Studio, với `USE_DRIVE = False`. Tôi chưa chạy thử trường hợp này.

## Bước 1. Kiểm tra GPU

In [ ]:
#@title Kiểm tra GPU và PyTorch
import sys
import torch

print("Python", sys.version.split()[0], "| PyTorch", torch.__version__)
if torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability(0)
    _mem = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU: {torch.cuda.get_device_name(0)} | {_mem:.0f} GB | compute capability {_cap[0]}.{_cap[1]}"
          f" -> huấn luyện bằng {'bf16' if _cap[0] >= 8 else 'fp16'}")
else:
    print("⚠️ KHÔNG có GPU. Vào Runtime → Change runtime type → chọn T4 GPU → Save, rồi chạy lại từ đầu.\n"
          "   (Vẫn chạy được trên CPU, nhưng huấn luyện rất chậm.)")
_v = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
if _v < (2, 6):
    print("⚠️ PyTorch < 2.6: transformers mới sẽ từ chối nạp pytorch_model.bin của ViDeBERTa.\n"
          "   Chạy:  !pip install -U 'torch>=2.6'  rồi Runtime → Restart session và chạy lại từ đầu.")

## Bước 2. Cài thư viện (khoảng 1–2 phút)

- Notebook được viết cho **Transformers 4.x**, nên cố định phiên bản `<5` để tránh khác biệt API.
- Pip có thể in cảnh báo xung đột với các gói khác có sẵn trên Colab. Cứ bỏ qua, miễn là dòng cuối của ô in ra được phiên bản thư viện.
- Nếu trước đó bạn đã chạy notebook trong **cùng phiên** và pip vừa đổi phiên bản `transformers`: chọn **Runtime → Restart session**, rồi chạy lại từ Bước 1.

In [ ]:
#@title Cài thư viện
!pip install -q "transformers>=4.46,<5" "datasets>=2.18" seqeval "accelerate>=0.26" sentencepiece fastapi "uvicorn[standard]"
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

## Bước 3. Cấu hình

Trên Colab, các tham số hiện thành **ô nhập** ở bên phải. Lần đầu nên giữ nguyên.

| Tham số | Ý nghĩa |
|---|---|
| `USE_DRIVE` | Lưu mọi thứ lên Google Drive. Nếu tắt, mọi thứ nằm ở `/content` và **mất khi hết phiên** |
| `PROJECT_DIR` | Thư mục dự án trên Drive |
| `TOOLKIT_ZIP` | Để trống thì tự tìm `bo_cong_cu_ner_tien_hiep*.zip` (bản mới nhất) trong `PROJECT_DIR` |
| `DATA_SOURCE` | `demo`: dữ liệu mẫu tự sinh, 2.000 câu (chạy thử) · `export_mau`: file export Label Studio mẫu trong bộ công cụ (thử đúng quy trình thật) · `label_studio`: **file export của bạn** trong `PROJECT_DIR/exports/` · `jsonl`: **train/dev/test.jsonl của bạn** trong `PROJECT_DIR/data/` |
| `NUM_EPOCHS`, `BATCH_SIZE`, `LEARNING_RATE`, `MAX_LEN` | Tham số huấn luyện. T4 16 GB chạy tốt với 16 / 256 |
| `SAVE_CHECKPOINTS_TO_DRIVE` | Lưu checkpoint lên Drive để chạy tiếp khi mất kết nối. **Mỗi checkpoint khoảng 2 GB** và giữ tối đa 2 cái, nên chỉ bật khi dữ liệu lớn |
| `RESUME_FROM_DRIVE_CHECKPOINT` | Bật khi chạy lại sau lần bị mất kết nối: huấn luyện tiếp từ checkpoint mới nhất. **Phải dùng đúng dữ liệu cũ** |
| `RUN_API_TEST` | Chạy thử API ở Bước 11 |

In [ ]:
#@title Cấu hình { display-mode: "form" }
#@markdown **Lưu trữ**
USE_DRIVE = True  #@param {type:"boolean"}
PROJECT_DIR = "/content/drive/MyDrive/ner_tien_hiep"  #@param {type:"string"}
TOOLKIT_ZIP = ""  #@param {type:"string"}
#@markdown **Dữ liệu huấn luyện**
DATA_SOURCE = "demo"  #@param ["demo", "export_mau", "label_studio", "jsonl"]
#@markdown **Huấn luyện**
MODEL_NAME = "Fsoft-AIC/videberta-base"  #@param {type:"string"}
NUM_EPOCHS = 10  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
LEARNING_RATE = 3e-5  #@param {type:"number"}
MAX_LEN = 256  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
SAVE_CHECKPOINTS_TO_DRIVE = False  #@param {type:"boolean"}
RESUME_FROM_DRIVE_CHECKPOINT = False  #@param {type:"boolean"}
#@markdown **Kiểm tra sau khi huấn luyện**
RUN_API_TEST = True  #@param {type:"boolean"}

import os, re, json, random, inspect, glob, shutil, subprocess, time, gc
from collections import Counter
from datetime import datetime
import numpy as np
import pandas as pd

try:
    from google.colab import drive as _colab_drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if USE_DRIVE and IN_COLAB:
    _colab_drive.mount("/content/drive")
elif USE_DRIVE:
    print("ℹ️ Không chạy trên Colab: không mount Drive, dùng PROJECT_DIR như thư mục bình thường.")
if (not USE_DRIVE or not IN_COLAB) and PROJECT_DIR.startswith("/content/drive"):
    PROJECT_DIR = "/content/ner_tien_hiep" if IN_COLAB else os.path.abspath("ner_tien_hiep_project")
    print("ℹ️ PROJECT_DIR =", PROJECT_DIR, "(không phải Drive: dữ liệu sẽ mất khi hết phiên)" if IN_COLAB else "")

def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(s))]

try:
    from zoneinfo import ZoneInfo
    _now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
except Exception:
    _now = datetime.now()
RUN_ID = _now.strftime("%Y%m%d-%H%M")

BASE_DIR = "/content" if IN_COLAB else os.path.abspath(".")
WORK = os.path.join(BASE_DIR, "work")                  # ổ đĩa của phiên Colab: nhanh, nhưng tạm thời
RUN_DIR = os.path.join(PROJECT_DIR, "runs", RUN_ID)
REPORT_DIR = os.path.join(RUN_DIR, "bao_cao")
DATA_DIR = os.path.join(WORK, "data")
FINAL_DIR = os.path.join(WORK, "ner-final")
MODEL_STORE = os.path.join(PROJECT_DIR, "models", f"ner-final-{RUN_ID}")
EARLY_STOPPING_PATIENCE = 2
USE_DEMO_DATA = DATA_SOURCE == "demo"
for _d in (PROJECT_DIR, RUN_DIR, REPORT_DIR, WORK):
    os.makedirs(_d, exist_ok=True)
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)                            # không lẫn dữ liệu của lần chạy trước
os.makedirs(DATA_DIR)

# Checkpoint: trên Drive (chạy tiếp được khi mất kết nối) hoặc trên ổ đĩa phiên (nhanh, mất khi hết phiên)
_ck_root = os.path.join(PROJECT_DIR, "checkpoints")
OUTPUT_DIR = os.path.join(WORK, "ner-out")
if SAVE_CHECKPOINTS_TO_DRIVE or RESUME_FROM_DRIVE_CHECKPOINT:
    OUTPUT_DIR = os.path.join(_ck_root, RUN_ID)
    if RESUME_FROM_DRIVE_CHECKPOINT:
        _prev = sorted((d for d in glob.glob(os.path.join(_ck_root, "*")) if glob.glob(os.path.join(d, "checkpoint-*"))),
                       key=natkey)
        if _prev:
            OUTPUT_DIR = _prev[-1]
            print("↻ Sẽ huấn luyện tiếp từ:", OUTPUT_DIR)
        else:
            print("⚠️ RESUME_FROM_DRIVE_CHECKPOINT = True nhưng không có checkpoint nào trong", _ck_root)
    SAVE_CHECKPOINTS_TO_DRIVE = True

ENV = dict(os.environ, PYTHONUTF8="1", PYTHONIOENCODING="utf-8")

def copy_dir(src, dst):
    """Chép cả thư mục, chỉ chép nội dung file (không chép quyền / thời gian sửa).
    Ổ mạng như Google Drive có thể từ chối ghi các thông tin đó; shutil.copytree khi ấy sẽ báo lỗi."""
    for root, _, fs in os.walk(src):
        out = os.path.join(dst, os.path.relpath(root, src))
        os.makedirs(out, exist_ok=True)
        for f in fs:
            shutil.copyfile(os.path.join(root, f), os.path.join(out, f))

def sh(args, cwd=None, env=None, check=True):
    """Chạy một lệnh và in kết quả ngay trong notebook."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    p = subprocess.Popen(args, cwd=cwd, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}). Xem thông báo ngay phía trên.")
    return p.returncode

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"RUN_ID = {RUN_ID} | dữ liệu: {DATA_SOURCE} | thiết bị: {DEVICE}")
print("Kết quả lần chạy này:", RUN_DIR)
print("Checkpoint:", OUTPUT_DIR)

## Bước 4. Lấy bộ công cụ

Ô này tìm file `bo_cong_cu_ner_tien_hiep*.zip` trong `PROJECT_DIR` (hoặc thư mục cha của nó), lấy **bản mới nhất** rồi giải nén vào `/content`. Nếu không thấy file, Colab sẽ hiện nút để bạn tải file lên.

In [ ]:
#@title Tìm và giải nén bộ công cụ
import zipfile, sys

def _find_toolkit_zip():
    if TOOLKIT_ZIP:
        if not os.path.exists(TOOLKIT_ZIP):
            raise FileNotFoundError(f"Không thấy TOOLKIT_ZIP = {TOOLKIT_ZIP}")
        return TOOLKIT_ZIP
    cands = []
    for d in {PROJECT_DIR, os.path.dirname(PROJECT_DIR), BASE_DIR}:
        cands += glob.glob(os.path.join(d, "bo_cong_cu_ner_tien_hiep*.zip"))
    if cands:
        return max(cands, key=lambda p: (natkey(os.path.basename(p)), os.path.getmtime(p)))
    if IN_COLAB:
        from google.colab import files as _colab_files
        print(f"Không thấy file bộ công cụ trong {PROJECT_DIR}. Chọn file .zip để tải lên:")
        up = _colab_files.upload()
        if not up:
            raise FileNotFoundError("Chưa chọn file nào.")
        name = next(iter(up))
        dst = os.path.join(PROJECT_DIR, os.path.basename(name))
        with open(dst, "wb") as f:
            f.write(up[name])
        return dst
    raise FileNotFoundError(f"Không thấy bo_cong_cu_ner_tien_hiep*.zip trong {PROJECT_DIR}")

_zip = _find_toolkit_zip()
print("Bộ công cụ:", _zip)
with zipfile.ZipFile(_zip) as z:
    _roots = {n.split("/")[0] for n in z.namelist() if "/" in n}
    if "bo_cong_cu_ner_tien_hiep" not in _roots:
        raise ValueError("File zip không đúng cấu trúc: thiếu thư mục bo_cong_cu_ner_tien_hiep/")
    TOOLKIT_DIR = os.path.join(BASE_DIR, "bo_cong_cu_ner_tien_hiep")
    if os.path.isdir(TOOLKIT_DIR):
        shutil.rmtree(TOOLKIT_DIR)
    z.extractall(BASE_DIR)

_need = ["xay_dung_du_lieu.py", "gan_nhan_nhap_tu_dien.py", "trien_khai/ner_engine.py",
         "trien_khai/xu_ly_file.py", "trien_khai/gop_ten.py", "trien_khai/api.py"]
_miss = [f for f in _need if not os.path.exists(os.path.join(TOOLKIT_DIR, f))]
if _miss:
    raise FileNotFoundError(f"Bộ công cụ thiếu {_miss}. Hãy dùng bản 1.2 trở lên.")
os.chdir(TOOLKIT_DIR)
sys.path.insert(0, os.path.join(TOOLKIT_DIR, "trien_khai"))
_ver = re.search(r"\*\*Phiên bản ([^*]+)\*\*", open(os.path.join(TOOLKIT_DIR, "README.md"), encoding="utf-8").read())
print("✅ Đã giải nén vào", TOOLKIT_DIR, "|", _ver.group(1) if _ver else "")

## Bước 5. Chuẩn bị dữ liệu

| `DATA_SOURCE` | Ô này làm gì |
|---|---|
| `demo` | Không làm gì; dữ liệu mẫu được sinh ở Bước 6.2 |
| `export_mau` | Chạy `xay_dung_du_lieu.py split` trên file export mẫu (600 câu), giống hệt quy trình thật |
| `label_studio` | Chạy `split` trên **mọi file `.json`** trong `PROJECT_DIR/exports/`. Bạn tải lên các file export (định dạng **JSON**) của các vòng gán nhãn; câu trùng giữa các file được tự loại |
| `jsonl` | Sao chép `train.jsonl` / `dev.jsonl` / `test.jsonl` từ `PROJECT_DIR/data/`. Có thể chỉ có `train.jsonl`, hoặc `all.jsonl` để tự chia |

Dữ liệu đã dùng được lưu bản sao vào `runs/<RUN_ID>/data/` để sau này biết mô hình được huấn luyện trên dữ liệu nào.

In [ ]:
#@title Chuẩn bị dữ liệu
if DATA_SOURCE == "demo":
    print("Dùng dữ liệu mẫu tự sinh (Bước 6.2).")
elif DATA_SOURCE in ("export_mau", "label_studio"):
    if DATA_SOURCE == "export_mau":
        _exports = [os.path.join(TOOLKIT_DIR, "vi_du_export", "export_mau.json")]
        _target = 100      # file mẫu nhỏ: chỉ tiêu thấp để báo cáo dễ đọc
    else:
        _exp_dir = os.path.join(PROJECT_DIR, "exports")
        os.makedirs(_exp_dir, exist_ok=True)
        _exports = sorted(glob.glob(os.path.join(_exp_dir, "*.json")), key=natkey)
        _target = 300
        if not _exports:
            raise FileNotFoundError(f"Chưa có file export nào trong {_exp_dir}.\n"
                                    "Trong Label Studio: Export → JSON, rồi tải file lên thư mục này.")
    print("File export:", *[os.path.basename(p) for p in _exports], sep="\n  ")
    sh([sys.executable, "xay_dung_du_lieu.py", "split", "--input", *_exports, "--out", DATA_DIR,
        "--target", _target], cwd=TOOLKIT_DIR)
elif DATA_SOURCE == "jsonl":
    _src = os.path.join(PROJECT_DIR, "data")
    _found = [n for n in ("train", "dev", "test", "all") if os.path.exists(os.path.join(_src, n + ".jsonl"))]
    if "train" not in _found and "all" not in _found:
        raise FileNotFoundError(f"Cần train.jsonl (hoặc all.jsonl) trong {_src}")
    for n in _found:
        shutil.copy(os.path.join(_src, n + ".jsonl"), DATA_DIR)
    print("Đã chép:", _found)
else:
    raise ValueError(f"DATA_SOURCE không hợp lệ: {DATA_SOURCE}")

if not USE_DEMO_DATA:
    copy_dir(DATA_DIR, os.path.join(RUN_DIR, "data"))

## Bước 6. Huấn luyện

Các ô dưới đây **giống notebook `ner_finetune_tieng_viet.ipynb`** trong bộ công cụ, nên kết quả hai bên khớp nhau.

### 6.1 Tách từ
Cách tách từ ở đây phải **giống hệt** lúc gán nhãn và lúc dự đoán.

In [ ]:
TOKEN_RE = re.compile(r"\w+(?:[.,/:\-]\w+)*|[^\w\s]", re.UNICODE)

def simple_tokenize(text):
    """Tách theo âm tiết + dấu câu (giữ nguyên số kiểu 5.000.000). PHẢI giống lúc dự đoán."""
    return TOKEN_RE.findall(text)

print(simple_tokenize("Tiêu Viêm rút Huyền Trọng Thước, thi triển Phật Nộ Hỏa Liên."))

### 6.2 Dữ liệu mẫu (chỉ khi `DATA_SOURCE = demo`)
Sinh 4 bộ truyện giả lập. Bộ thứ 4 có tên riêng gần như hoàn toàn mới và được dùng làm tập test, để đo khả năng **nhận ra tên chưa gặp nhờ ngữ cảnh**. Dữ liệu sinh từ mẫu câu nên điểm số sẽ cao hơn dữ liệu thật.

In [ ]:
# ---------------- Kho từ để sinh tên ----------------
SYL = ("Vân Lam Thanh Liên Huyền Trọng Thiên Địa Huyết Ma Cổ Long Phượng Hạo Nguyệt Tinh Thần Vương "
       "Bạch Hắc Kim Ngọc Bích Hồng Tử Linh Lôi Phong Hàn Băng Dực Phệ Lãng Xích Bát Cực Phật Nộ "
       "U Minh Thái Hư Cửu Vô Tịch Diệt Lưu Vạn Âm Dương Thủy Mộc Thổ Tuyết Sương Vũ Quang "
       "Ảnh Hồn Phách Tâm Ý Kiếp Luân Hồi Tiên Thánh Lăng Tiêu Diệp Tần Mặc Lâm Hoang Man Yêu").split()
HO  = ["Tiêu", "Nạp Lan", "Hàn", "Lâm", "Mộc", "Tô", "Mặc", "Thạch", "Cổ", "Lăng", "Diệp", "Tần",
       "Liễu", "Bạch", "Vương", "Hạ Hầu", "Mộ Dung", "Thượng Quan", "Âu Dương", "Lạc"]
TEN = ["Viêm", "Vũ", "Phong", "Hàn", "Lập", "Nguyệt", "Tuyết", "Thần", "Hạo", "Uyển", "Ngưng", "Dao",
       "Kiệt", "Vân", "Lam", "Thiên", "Mạc", "Trần", "Linh", "Huyên", "Nhiên", "Yên", "Vô Song", "Thanh Hà"]
HEAD = {
    "DIA_DANH": ["Sơn", "Thành", "Cốc", "Sơn Mạch", "Đế Quốc", "Trấn", "Hồ", "Sa Mạc", "Đảo"],
    "VU_KHI":   ["Kiếm", "Đao", "Thước", "Thương", "Cung", "Phiến", "Kích"],
    "CONG_PHAP": ["Quyết", "Chưởng", "Chỉ", "Quyền", "Băng", "Kinh", "Thức", "Trảm", "Ấn"],
    "DI_HOA":   ["Hỏa", "Hỏa", "Viêm"],
    "THE_LUC":  ["Tông", "Môn", "Điện", "Các", "Cung", "Học Viện", "Thương Hội", "Tháp"],
}
# Từ viết hoa nhưng KHÔNG phải thực thể (quy tắc R1, ca khó 9, 13, 16)
O_SLOTS = {
    "CANH_GIOI": ["Đấu Giả", "Đấu Sư", "Đại Đấu Sư", "Đấu Linh", "Đấu Vương", "Đấu Hoàng", "Đấu Tông", "Đấu Tôn"],
    "CAP": ["Thiên giai", "Địa giai", "Huyền giai", "Hoàng giai"],
    "DAN": ["Tụ Khí Tán", "Hồi Khí Đan", "Bồi Nguyên Đan", "Trúc Cơ Linh Dịch", "Tam Văn Thanh Linh Đan"],
}
ENTITY_LABELS = ["NHAN_VAT", "DIA_DANH", "VU_KHI", "CONG_PHAP", "DI_HOA", "THE_LUC"]

def gen_name(label, rng):
    if label == "NHAN_VAT":
        if rng.random() < 0.08:
            return rng.choice(SYL) + " Lão"                       # danh xưng cố định, vd. Dược Lão
        return rng.choice(HO) + " " + " ".join(rng.sample(TEN, rng.choice([1, 1, 2])))
    if label == "THE_LUC" and rng.random() < 0.25:
        return rng.choice(HO) + " gia"                            # gia tộc: lấy cả chữ "gia"
    if label == "CONG_PHAP" and rng.random() < 0.3:               # chiêu thức có chữ "Hỏa" vẫn là công pháp
        return " ".join(rng.sample(SYL, 2)) + " Hỏa " + rng.choice(["Liên", "Luân", "Quyền", "Chưởng"])
    n_syl = {"DIA_DANH": [2], "VU_KHI": [2], "CONG_PHAP": [1, 2, 3], "DI_HOA": [2, 3], "THE_LUC": [2]}[label]
    return " ".join(rng.sample(SYL, rng.choice(n_syl))) + " " + rng.choice(HEAD[label])

NOVEL_SIZES = {"NHAN_VAT": 30, "DIA_DANH": 12, "VU_KHI": 10, "CONG_PHAP": 14, "DI_HOA": 7, "THE_LUC": 10}
SEED_NAMES = {   # tên thật trong tài liệu hướng dẫn, chỉ đưa vào bộ truyện 1
    "NHAN_VAT": ["Tiêu Viêm", "Nạp Lan Yên Nhiên", "Huân Nhi", "Dược Lão", "Vân Vận"],
    "DIA_DANH": ["Vân Lam Sơn", "Ô Thản Thành", "Ma Thú Sơn Mạch", "Gia Mã Đế Quốc"],
    "VU_KHI": ["Huyền Trọng Thước", "Tử Vân Kiếm"],
    "CONG_PHAP": ["Phần Quyết", "Bát Cực Băng", "Phật Nộ Hỏa Liên", "Dực Hỏa Phệ Lãng Xích"],
    "DI_HOA": ["Thanh Liên Địa Tâm Hỏa", "Cốt Linh Lãnh Hỏa", "Vẫn Lạc Tâm Viêm"],
    "THE_LUC": ["Vân Lam Tông", "Nạp Lan gia", "Tiêu gia", "Già Nam Học Viện"],
}

def make_novel_names(rng, seed_names=None):
    used, names = set(), {}
    for lab in ENTITY_LABELS:
        lst = list((seed_names or {}).get(lab, []))
        used.update(lst)
        while len(lst) < NOVEL_SIZES[lab]:
            n = gen_name(lab, rng)
            if n not in used:
                used.add(n); lst.append(n)
        names[lab] = lst
    return names

# (mẫu câu, trọng số). {NHAN_VAT}… = thực thể; {CANH_GIOI}/{CAP}/{DAN} = chữ viết hoa nhưng gán O
TEMPLATES = [
    ("{NHAN_VAT} ngồi xếp bằng trên tảng đá lớn tại đỉnh {DIA_DANH}, tay nắm chặt {VU_KHI}.", 3),
    ("Hắn vận chuyển đấu khí, thi triển {CONG_PHAP} hướng thẳng về phía {NHAN_VAT}.", 4),
    ("Luồng hỏa diễm màu xanh lục từ {DI_HOA} bùng phát dữ dội.", 2),
    ("Đệ tử {THE_LUC} đồng loạt lùi lại khi {NHAN_VAT} xuất hiện.", 3),
    ("“{NHAN_VAT} ca ca, huynh lại muốn vào {DIA_DANH} sao?” {NHAN_VAT} hỏi.", 3),
    ("Lão giả áo đen thi triển {CAP} đấu kỹ {CONG_PHAP}, đánh thẳng vào ngực thiếu niên.", 3),
    ("Sau khi luyện hóa {DI_HOA}, thực lực của {NHAN_VAT} đột phá lên {CANH_GIOI}.", 3),
    ("Tông chủ {NHAN_VAT} chậm rãi bước ra từ đại điện của {THE_LUC}.", 3),
    ("Hắn nuốt một viên {DAN}, rồi rút {VU_KHI} từ trong nạp giới ra.", 3),
    ("{THE_LUC} vốn là một trong ba đại thế lực của {DIA_DANH}.", 3),
    ("{NHAN_VAT} và {NHAN_VAT} sóng vai rời khỏi {DIA_DANH}.", 3),
    ("“{NHAN_VAT}, đi thôi!” {NHAN_VAT} khẽ gọi.", 2),
    ("Ba năm sau, ta nhất định sẽ lên {THE_LUC} tìm {NHAN_VAT}!", 2),
    ("Thanh {VU_KHI} trong tay {NHAN_VAT} rung lên bần bật.", 2),
    ("Hắn dùng {DI_HOA} thi triển {CONG_PHAP}, cả {DIA_DANH} rung chuyển.", 3),
    ("Nghe nói {DI_HOA} đang ẩn sâu dưới lòng {DIA_DANH}.", 2),
    ("{NHAN_VAT} thầm vận chuyển {CONG_PHAP}, đấu khí trong cơ thể cuồn cuộn.", 3),
    ("Người của {THE_LUC} đã tới {DIA_DANH}.", 2),
    ("{NHAN_VAT} tiền bối, vãn bối là đệ tử {THE_LUC}.", 2),
    ("Một kiếm chém xuống, {VU_KHI} của {NHAN_VAT} để lại vết nứt dài trên mặt đất.", 2),
    # Câu không có thực thể (mẫu âm, quy tắc R8)
    ("Gió thổi qua sườn núi, mây trắng lững lờ trôi.", 1),
    ("Hắn trầm mặc hồi lâu, rồi khẽ thở dài.", 1),
    ("Đấu khí trong cơ thể hắn đã đạt tới {CANH_GIOI} đỉnh phong.", 2),
    ("Một con ma thú cấp ba lao ra từ bụi rậm.", 1),
    ("Hỏa diễm bốc lên ngùn ngụt, nhiệt độ xung quanh tăng vọt.", 1),
    ("Lão giả cười lạnh: “Tiểu tử, ngươi chán sống rồi!”", 1),
    ("Nàng đem {DAN} cất vào nạp giới, ánh mắt tràn đầy ôn nhu.", 1),
]

def make_example(template, names, rng):
    tokens, tags, used = [], [], set()
    for part in re.split(r"(\{[A-Z_]+\})", template):
        if not part:
            continue
        m = re.fullmatch(r"\{([A-Z_]+)\}", part)
        if m and m.group(1) in O_SLOTS:
            words = simple_tokenize(rng.choice(O_SLOTS[m.group(1)]))
            tokens += words; tags += ["O"] * len(words)
        elif m:
            label = m.group(1)
            value = rng.choice([n for n in names[label] if n not in used] or names[label])
            used.add(value)
            words = simple_tokenize(value)
            tokens += words
            tags += [f"B-{label}"] + [f"I-{label}"] * (len(words) - 1)
        else:
            words = simple_tokenize(part)
            tokens += words; tags += ["O"] * len(words)
    return {"tokens": tokens, "ner_tags": tags}

NOVELS = {"truyen_1": 550, "truyen_2": 550, "truyen_3": 550, "truyen_4_test": 350}
DEMO_TRAIN = os.path.join(DATA_DIR, "demo_train.jsonl")
DEMO_TEST  = os.path.join(DATA_DIR, "demo_test.jsonl")

if USE_DEMO_DATA:
    rng = random.Random(SEED)
    tpls, weights = zip(*TEMPLATES)
    novel_names, rows = {}, {"train": [], "test": []}
    for novel, n_sent in NOVELS.items():
        novel_names[novel] = make_novel_names(rng, SEED_NAMES if novel == "truyen_1" else None)
        split = "test" if novel.endswith("_test") else "train"
        for _ in range(n_sent):
            ex = make_example(rng.choices(tpls, weights)[0], novel_names[novel], rng)
            ex["novel"] = novel
            rows[split].append(ex)
    for path, split in ((DEMO_TRAIN, "train"), (DEMO_TEST, "test")):
        with open(path, "w", encoding="utf-8") as f:
            for ex in rows[split]:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    train_names = {n for nv, d in novel_names.items() if not nv.endswith("_test") for v in d.values() for n in v}
    test_names = {n for v in novel_names["truyen_4_test"].values() for n in v}
    print(f"Đã tạo: {DEMO_TRAIN} ({len(rows['train'])} câu), {DEMO_TEST} ({len(rows['test'])} câu)")
    print(f"Tên trong tập test chưa từng xuất hiện ở train: {len(test_names - train_names)}/{len(test_names)}")
    ex = rows["train"][0]
    print(" ".join(f"{w}/{t}" if t != "O" else w for w, t in zip(ex["tokens"], ex["ner_tags"])))

### 6.3 Đọc dữ liệu và chia train / dev / test

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def read_conll(path):
    sents, toks, tags = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("-DOCSTART-"):
                if toks:
                    sents.append({"tokens": toks, "ner_tags": tags})
                    toks, tags = [], []
                continue
            parts = line.split()
            toks.append(parts[0]); tags.append(parts[-1])
    if toks:
        sents.append({"tokens": toks, "ner_tags": tags})
    return sents

def read_any(path):
    return read_conll(path) if path.endswith((".conll", ".txt")) else read_jsonl(path)

def find_file(name):
    for ext in (".jsonl", ".conll", ".txt"):
        p = os.path.join(DATA_DIR, name + ext)
        if os.path.exists(p):
            return p
    return None

def split_off(data, ratio, seed=SEED):
    data = data[:]
    random.Random(seed).shuffle(data)
    n = max(1, int(len(data) * ratio))
    return data[n:], data[:n]

if USE_DEMO_DATA:
    files = {"train": DEMO_TRAIN, "test": DEMO_TEST}   # test = bộ truyện riêng, tên mới
else:
    files = {name: find_file(name) for name in ("train", "dev", "test", "all")}
    files = {k: v for k, v in files.items() if v}
    if not files:
        raise FileNotFoundError(f"Không tìm thấy train/all (.jsonl/.conll) trong '{DATA_DIR}'")
print("File dữ liệu:", files)

raw = {k: read_any(v) for k, v in files.items()}
train = raw.get("train", raw.get("all"))
dev, test = raw.get("dev"), raw.get("test")
if test is None:
    train, test = split_off(train, 0.1)
if dev is None:
    train, dev = split_off(train, 0.1)

print(f"train={len(train)}  dev={len(dev)}  test={len(test)}")

### 6.4 Kiểm tra và thống kê dữ liệu
Nên có vài trăm thực thể mỗi nhãn trong train. Ô này cảnh báo các nhãn có dưới 50 thực thể.

In [ ]:
TAG_RE = re.compile(r"^(O|[BI]-.+)$")

def validate(data, name):
    clean, bad, fixed = [], 0, 0
    for ex in data:
        toks, tags = ex.get("tokens", []), list(ex.get("ner_tags", []))
        if not toks or len(toks) != len(tags) or not all(TAG_RE.match(t) for t in tags):
            bad += 1
            continue
        for j, tag in enumerate(tags):
            if tag.startswith("I-") and (j == 0 or tags[j - 1][2:] != tag[2:]):
                tags[j] = "B-" + tag[2:]
                fixed += 1
        clean.append({"tokens": toks, "ner_tags": tags})
    print(f"[{name}] giữ {len(clean)} câu | loại {bad} câu lỗi | sửa {fixed} nhãn I- -> B-")
    return clean

train, dev, test = validate(train, "train"), validate(dev, "dev"), validate(test, "test")

entity_types = sorted({t[2:] for ex in train + dev + test for t in ex["ner_tags"] if t != "O"})
LABELS = ["O"] + [f"{p}-{e}" for e in entity_types for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print("Các loại thực thể:", entity_types)

def count_entities(data):
    return Counter(t[2:] for ex in data for t in ex["ner_tags"] if t.startswith("B-"))

stats = pd.DataFrame({n: count_entities(d) for n, d in [("train", train), ("dev", dev), ("test", test)]})
stats = stats.reindex(entity_types).fillna(0).astype(int)
print(stats)

few = stats.index[stats["train"] < 50].tolist()
if few:
    print("⚠️ Các loại có ít hơn 50 mẫu trong train, nên bổ sung dữ liệu:", few)

### 6.5 Tokenize và căn nhãn theo subword
Lần đầu chạy, ô này tải mô hình gốc từ Hugging Face (khoảng 1 GB). Cảnh báo kiểu *HF_TOKEN… not set* có thể bỏ qua: mô hình công khai, không cần token.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# ViDeBERTa chỉ có tokenizer.json (không có spm.model) -> phải dùng fast tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert getattr(tokenizer, "is_fast", True), "Cần fast tokenizer để dùng word_ids()"

# Kiểm tra nhanh cách tokenizer chia subword
_demo = simple_tokenize("Tiêu Viêm thi triển Phật Nộ Hỏa Liên về phía đệ tử Vân Lam Tông")
_enc = tokenizer(_demo, is_split_into_words=True)
print(tokenizer.convert_ids_to_tokens(_enc["input_ids"]))
print(_enc.word_ids())

def tokenize_align(batch):
    enc = tokenizer(batch["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    all_labels = []
    for i, tags in enumerate(batch["ner_tags"]):
        prev, ids = None, []
        for w in enc.word_ids(batch_index=i):
            if w is None or w == prev:
                ids.append(-100)
            else:
                ids.append(label2id[tags[w]])
            prev = w
        all_labels.append(ids)
    enc["labels"] = all_labels
    return enc

raw_ds = DatasetDict({
    "train": Dataset.from_list(train),
    "validation": Dataset.from_list(dev),
    "test": Dataset.from_list(test),
})
tokenized = raw_ds.map(tokenize_align, batched=True, remove_columns=["tokens", "ner_tags"])
print(tokenized)

n_long = sum(len(tokenizer(ex["tokens"], is_split_into_words=True)["input_ids"]) > MAX_LEN
             for ex in train + dev + test)
if n_long:
    print(f"⚠️ {n_long} câu dài hơn {MAX_LEN} token sẽ bị cắt. Cân nhắc tách câu hoặc tăng MAX_LEN.")

### 6.6 Khởi tạo mô hình
Cảnh báo *Some weights … newly initialized* là **bình thường**: lớp phân loại cho 6 nhãn được tạo mới.

In [ ]:
from transformers import (AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def to_tag_lists(predictions, label_ids):
    preds = np.argmax(predictions, axis=-1)
    y_true, y_pred = [], []
    for p_row, l_row in zip(preds, label_ids):
        t, p = [], []
        for p_id, l_id in zip(p_row, l_row):
            if l_id == -100:
                continue
            t.append(id2label[int(l_id)])
            p.append(id2label[int(p_id)])
        y_true.append(t); y_pred.append(p)
    return y_true, y_pred

def compute_metrics(eval_pred):
    y_true, y_pred = to_tag_lists(eval_pred.predictions, eval_pred.label_ids)
    return {
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

# Lớp phân loại (classifier) được khởi tạo mới -> cảnh báo "Some weights ... newly initialized" là bình thường
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
print(f"Số tham số: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

# DeBERTaV3 ổn định hơn với bf16 (GPU Ampere trở lên: A100, A10G, L4, RTX 30xx/40xx); nếu không có thì dùng fp16
# Chỉ bật bf16 khi GPU hỗ trợ thật (compute capability >= 8.0). GPU T4 của Colab miễn phí là 7.5 -> dùng fp16.
# (Không dùng torch.cuda.is_bf16_supported(): bản PyTorch mới trả True cả trên T4 do giả lập, Trainer sẽ báo lỗi.)
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
USE_FP16 = torch.cuda.is_available() and not USE_BF16
print("bf16:", USE_BF16, "| fp16:", USE_FP16)

# Tương thích nhiều phiên bản transformers
ta_params = inspect.signature(TrainingArguments.__init__).parameters
strategy_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    bf16=USE_BF16,
    fp16=USE_FP16,
    report_to="none",
    seed=SEED,
    **{strategy_key: "epoch"},
)

tr_params = inspect.signature(Trainer.__init__).parameters
tok_kwarg = {"processing_class": tokenizer} if "processing_class" in tr_params else {"tokenizer": tokenizer}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    **tok_kwarg,
)

### 6.7 Huấn luyện

Thời gian tuỳ lượng dữ liệu và GPU. Với dữ liệu demo trên T4, ước tính vài phút.

**Nếu mất kết nối giữa chừng:**
1. Bật `SAVE_CHECKPOINTS_TO_DRIVE` ngay từ đầu.
2. Khi chạy lại, bật thêm `RESUME_FROM_DRIVE_CHECKPOINT`, giữ nguyên dữ liệu, rồi **Run all**.

In [ ]:
#@title Huấn luyện
_resume = None
if RESUME_FROM_DRIVE_CHECKPOINT:
    _cks = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")), key=natkey)
    if _cks:
        _resume = True
        print("↻ Tiếp tục từ", _cks[-1])
    else:
        print("⚠️ Không thấy checkpoint trong", OUTPUT_DIR, "-> huấn luyện từ đầu")
_t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=_resume)
TRAIN_SECONDS = time.time() - _t0
print(train_result.metrics)
DEV_METRICS = trainer.evaluate()
print(f"Thời gian huấn luyện: {TRAIN_SECONDS / 60:.1f} phút")
print("Kết quả tốt nhất trên dev:", DEV_METRICS)

## Bước 7. Đánh giá, lưu báo cáo và mô hình lên Drive

### 7.1 Đánh giá trên tập test
Hãy xem **F1 của từng nhãn**, đừng chỉ nhìn con số tổng. Bảng điểm được lưu vào `runs/<RUN_ID>/bao_cao/`.

In [ ]:
test_out = trainer.predict(tokenized["test"])
y_true, y_pred = to_tag_lists(test_out.predictions, test_out.label_ids)

print(f"Test  P={precision_score(y_true, y_pred):.4f}  R={recall_score(y_true, y_pred):.4f}  F1={f1_score(y_true, y_pred):.4f}\n")
print(classification_report(y_true, y_pred, digits=4))

_rep = classification_report(y_true, y_pred, output_dict=True)
TEST_REPORT = pd.DataFrame([{"nhan": k, "precision": round(v["precision"], 4), "recall": round(v["recall"], 4),
                             "f1": round(v["f1-score"], 4), "so_thuc_the": int(v["support"])}
                            for k, v in _rep.items()])
TEST_REPORT.to_csv(os.path.join(REPORT_DIR, "ket_qua_test.csv"), index=False, encoding="utf-8-sig")
with open(os.path.join(REPORT_DIR, "ket_qua_test.txt"), "w", encoding="utf-8") as f:
    f.write(classification_report(y_true, y_pred, digits=4))
TEST_METRICS = {"precision": float(precision_score(y_true, y_pred)), "recall": float(recall_score(y_true, y_pred)),
                "f1": float(f1_score(y_true, y_pred))}

### 7.2 Phân tích lỗi
Toàn bộ câu đoán sai được lưu vào `bao_cao/cau_doan_sai.csv` để đọc kỹ hoặc đưa vào vòng gán nhãn sau.

In [ ]:
errors = []
for ex, t, p in zip(test, y_true, y_pred):
    if t != p:
        errors.append((ex["tokens"][:len(t)], t, p))

print(f"{len(errors)}/{len(test)} câu có ít nhất một lỗi\n")
for words, t, p in errors[:10]:
    print(" ".join(words))
    for w, a, b in zip(words, t, p):
        if a != b:
            print(f"    {w:<22} đúng={a:<14} dự đoán={b}")
    print("-" * 70)

_rows = []
for words, t, p in errors:
    for w, a, b in zip(words, t, p):
        if a != b:
            _rows.append({"cau": " ".join(words), "tu": w, "nhan_dung": a, "du_doan": b})
pd.DataFrame(_rows, columns=["cau", "tu", "nhan_dung", "du_doan"]).to_csv(
    os.path.join(REPORT_DIR, "cau_doan_sai.csv"), index=False, encoding="utf-8-sig")
print(f"Đã lưu {len(_rows)} từ đoán sai -> {REPORT_DIR}/cau_doan_sai.csv")

### 7.3 Lưu mô hình
Mô hình được lưu vào `/content/work/ner-final` rồi **chép lên Drive** tại `models/ner-final-<RUN_ID>/`. Kèm theo là `model_info.json`, ghi lại dữ liệu, tham số, điểm số và phiên bản thư viện.

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
with open(os.path.join(FINAL_DIR, "labels.json"), "w", encoding="utf-8") as f:
    json.dump(LABELS, f, ensure_ascii=False, indent=2)
print("Đã lưu mô hình vào:", FINAL_DIR)

import transformers
MODEL_INFO = {
    "run_id": RUN_ID, "base_model": MODEL_NAME, "data_source": DATA_SOURCE, "labels": LABELS,
    "max_len": MAX_LEN, "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "seed": SEED, "n_sentences": {"train": len(train), "dev": len(dev), "test": len(test)},
    "dev": {k: float(v) for k, v in DEV_METRICS.items() if isinstance(v, (int, float))},
    "test": TEST_METRICS,
    "test_f1_per_label": {r["nhan"]: r["f1"] for _, r in TEST_REPORT.iterrows() if r["nhan"] in set(entity_types)},
    "train_minutes": round(globals().get("TRAIN_SECONDS", 0) / 60, 1),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "torch": torch.__version__, "transformers": transformers.__version__,
}
for _p in (os.path.join(FINAL_DIR, "model_info.json"), os.path.join(REPORT_DIR, "model_info.json")):
    with open(_p, "w", encoding="utf-8") as f:
        json.dump(MODEL_INFO, f, ensure_ascii=False, indent=2)
print("Đang chép mô hình lên", MODEL_STORE, "...")
copy_dir(FINAL_DIR, MODEL_STORE)
print(f"✅ Mô hình: {MODEL_STORE}\n   Test F1 = {TEST_METRICS['f1']:.4f}")

## Bước 8. Chạy thử mô hình bằng `ner_engine.py`

Từ đây trở đi, notebook dùng **đúng các script triển khai** trong thư mục `trien_khai/`, tức là giống hệt cách mô hình sẽ chạy thật.

- Để trống `MODEL_FOR_TEST` thì dùng mô hình vừa huấn luyện.
- Hoặc điền thư mục một mô hình đã lưu, ví dụ `/content/drive/MyDrive/ner_tien_hiep/models/ner-final-20261117-1240`.

In [ ]:
#@title Chọn mô hình và chạy thử câu mẫu
MODEL_FOR_TEST = ""  #@param {type:"string"}

for _n in ("trainer", "model"):            # giải phóng bộ nhớ GPU của bước huấn luyện
    globals().pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TEST_MODEL = MODEL_FOR_TEST or FINAL_DIR
if not os.path.exists(os.path.join(TEST_MODEL, "config.json")):
    raise FileNotFoundError(f"Không thấy mô hình ở {TEST_MODEL}. Nếu phiên Colab đã khởi động lại, "
                            "hãy điền MODEL_FOR_TEST là thư mục mô hình trên Drive (PROJECT_DIR/models/...).")
_info_p = os.path.join(TEST_MODEL, "model_info.json")
MAX_LEN_TEST = json.load(open(_info_p, encoding="utf-8")).get("max_len", MAX_LEN) if os.path.exists(_info_p) else MAX_LEN

from ner_engine import NerEngine
eng = NerEngine(TEST_MODEL, max_len=MAX_LEN_TEST)
print(eng.info())

for s in [
    "Tiêu Viêm ngồi xếp bằng trên tảng đá lớn tại đỉnh Vân Lam Sơn, tay nắm chặt Huyền Trọng Thước. "
    "Hắn vận chuyển đấu khí, thi triển Dực Hỏa Phệ Lãng Xích hướng thẳng về phía Nạp Lan Yên Nhiên. "
    "Luồng hỏa diễm màu xanh lục từ Thanh Liên Địa Tâm Hỏa bùng phát dữ dội.",
    "Lăng Thiên Hạo rút Bích Lạc Kiếm, thi triển Cửu U Hàn Băng Chưởng về phía đệ tử Thiên Kiếm Môn trên đỉnh Tử Hà Sơn.",
    "Sau khi luyện hóa Cửu U Minh Viêm, Mộ Dung Tuyết đột phá lên Đấu Tông.",
]:
    print("\n" + s)
    for e in eng.extract(s)["entities"]:
        print(f"   [{e['label']:<10}] {e['text']:<28} điểm {e['score']:.3f}")

## Bước 9. Chạy mô hình trên cả thư mục truyện

- Nếu `PROJECT_DIR/truyen/` có file `.txt` thì dùng truyện của bạn. Nếu không, dùng `truyen_mau/` trong bộ công cụ.
- Nếu có `PROJECT_DIR/tu_dien.json` thì dùng thêm từ điển để bổ sung tên mô hình bỏ sót.

Kết quả nằm trong `runs/<RUN_ID>/ket_qua/`:
- `entities.csv`: từng lần xuất hiện.
- `tong_hop.csv`: danh sách tên không trùng lặp.
- `html/`: văn bản tô màu thực thể.

In [ ]:
#@title Xử lý thư mục truyện
from IPython.display import HTML, display

_own = os.path.join(PROJECT_DIR, "truyen")
INPUT_NOVELS = _own if glob.glob(os.path.join(_own, "**", "*.txt"), recursive=True) else os.path.join(TOOLKIT_DIR, "truyen_mau")
DICT_PATH = os.path.join(PROJECT_DIR, "tu_dien.json")
OUT_KQ = os.path.join(RUN_DIR, "ket_qua")
_args = [sys.executable, "trien_khai/xu_ly_file.py", "--model", TEST_MODEL, "--input", INPUT_NOVELS,
         "--out", OUT_KQ, "--html", "--max-len", MAX_LEN_TEST]
if os.path.exists(DICT_PATH):
    _args += ["--dict", DICT_PATH]
print("Truyện:", INPUT_NOVELS)
sh(_args, cwd=TOOLKIT_DIR)

display(pd.read_csv(os.path.join(OUT_KQ, "tong_hop.csv")).sort_values("count", ascending=False).head(15))
_htmls = sorted(glob.glob(os.path.join(OUT_KQ, "html", "*.html")), key=natkey)
if _htmls:
    print("Xem trước:", os.path.basename(_htmls[0]))
    display(HTML(open(_htmls[0], encoding="utf-8").read()))

## Bước 10. Gộp tên đồng nhất

Gom các cách gọi của cùng một thực thể, ví dụ "Nạp Lan Yên Nhiên" và "Yên Nhiên", về **một mã thực thể**. Nếu có `PROJECT_DIR/bi_danh.json` (từ những lần duyệt trước) thì file này được dùng luôn.

Kết quả nằm trong `runs/<RUN_ID>/lien_ket/`:
- `thuc_the.csv`: danh sách thực thể.
- `can_duyet.csv`: các dòng cần người duyệt.
- `xung_dot_nhan.csv`: tên bị gán nhiều nhãn.

In [ ]:
#@title Gộp tên
ALIAS_PATH = os.path.join(PROJECT_DIR, "bi_danh.json")
OUT_LK = os.path.join(RUN_DIR, "lien_ket")

def run_gop():
    a = [sys.executable, "trien_khai/gop_ten.py", "gop", "--input", os.path.join(OUT_KQ, "entities.jsonl"), "--out", OUT_LK]
    if os.path.exists(ALIAS_PATH):
        a += ["--alias", ALIAS_PATH]
    sh(a, cwd=TOOLKIT_DIR)

run_gop()
print("\nThực thể xuất hiện nhiều nhất:")
display(pd.read_csv(os.path.join(OUT_LK, "thuc_the.csv")).sort_values("so_lan", ascending=False)
        [["entity_id", "label", "ten_chuan", "so_lan", "cac_bien_the"]].head(15))
_cd = pd.read_csv(os.path.join(OUT_LK, "can_duyet.csv"))
print(f"\nCần duyệt: {len(_cd)} dòng (5 dòng đầu):")
display(_cd[["uu_tien", "loai", "ten_a", "ten_b", "ly_do"]].head(5))

### 10b. (Tuỳ chọn) Áp dụng kết quả duyệt

1. Trên Drive, mở `runs/<RUN_ID>/lien_ket/can_duyet.csv`. Ở cột `quyet_dinh`, điền `y` (gộp), `n` (không gộp), hoặc để trống. Xem mục D.4 trong `trien_khai/README.md`.
   - **Excel:** lưu bằng **CSV UTF-8 (Comma delimited)**.
   - **Google Sheets:** File → Download → CSV, rồi tải file lên lại Drive.
2. Đặt tên file đã điền là `can_duyet_da_dien.csv` và để trong `PROJECT_DIR`.
3. Bật `APPLY_REVIEW` rồi chạy ô dưới. Quyết định được ghi vào `PROJECT_DIR/bi_danh.json` và dùng tự động cho những lần chạy sau.

In [ ]:
#@title Áp dụng kết quả duyệt
APPLY_REVIEW = False  #@param {type:"boolean"}
REVIEW_FILE = ""  #@param {type:"string"}
#@markdown Để trống `REVIEW_FILE` = `PROJECT_DIR/can_duyet_da_dien.csv`
if APPLY_REVIEW:
    _rf = REVIEW_FILE or os.path.join(PROJECT_DIR, "can_duyet_da_dien.csv")
    if not os.path.exists(_rf):
        raise FileNotFoundError(f"Không thấy {_rf}")
    sh([sys.executable, "trien_khai/gop_ten.py", "ap-dung", "--review", _rf, "--alias", ALIAS_PATH], cwd=TOOLKIT_DIR)
    run_gop()
else:
    print("Bỏ qua (APPLY_REVIEW = False). File cần duyệt:", os.path.join(OUT_LK, "can_duyet.csv"))

## Bước 11. Chạy thử API

Ô này khởi động `api.py` **ngay trong phiên Colab**, chỉ nghe ở địa chỉ nội bộ `127.0.0.1` và có API key ngẫu nhiên. Sau đó ô gọi thử bằng `goi_api_mau.py`, kiểm tra rằng request thiếu key bị từ chối, rồi **tắt API**.

Đây chỉ là bước kiểm tra. **Không mở API này ra Internet.** Khi triển khai thật, dùng Docker hoặc SageMaker theo `trien_khai/README.md`.

In [ ]:
#@title Chạy thử API
import socket, secrets, urllib.request, urllib.error

if not RUN_API_TEST:
    print("Bỏ qua (RUN_API_TEST = False)")
else:
    with socket.socket() as _s:
        _s.bind(("127.0.0.1", 0)); _port = _s.getsockname()[1]
    _url = f"http://127.0.0.1:{_port}"
    _key = secrets.token_urlsafe(16)
    _env = dict(ENV, MODEL_DIR=TEST_MODEL, API_KEY=_key, MAX_LEN=str(MAX_LEN_TEST))
    if os.path.exists(DICT_PATH):
        _env["DICT_PATH"] = DICT_PATH
    _log_path = os.path.join(RUN_DIR, "api_log.txt")
    _log = open(_log_path, "w", encoding="utf-8")
    _api = subprocess.Popen([sys.executable, "-m", "uvicorn", "api:app", "--host", "127.0.0.1", "--port", str(_port)],
                            cwd=os.path.join(TOOLKIT_DIR, "trien_khai"), env=_env, stdout=_log, stderr=subprocess.STDOUT)
    try:
        print("Đang khởi động API (nạp mô hình)...")
        _ok = False
        for _ in range(90):
            if _api.poll() is not None:
                break
            try:
                urllib.request.urlopen(_url + "/health", timeout=2)
                _ok = True
                break
            except Exception:
                time.sleep(2)
        if not _ok:
            _log.flush()
            print(open(_log_path, encoding="utf-8").read()[-3000:])
            raise RuntimeError("API không khởi động được. Xem log ở trên (đã lưu ở api_log.txt).")
        sh([sys.executable, "goi_api_mau.py", "--url", _url, "--key", _key], cwd=os.path.join(TOOLKIT_DIR, "trien_khai"))
        _req = urllib.request.Request(_url + "/extract", data=json.dumps({"text": "Tiêu Viêm cười."}).encode("utf-8"),
                                      headers={"Content-Type": "application/json"})
        try:
            urllib.request.urlopen(_req, timeout=30)
            print("⚠️ Request KHÔNG có API key vẫn được chấp nhận: kiểm tra lại cấu hình API_KEY")
        except urllib.error.HTTPError as e:
            print(f"\nRequest không có API key -> HTTP {e.code} " + ("✅ (đúng)" if e.code == 401 else "⚠️"))
    finally:
        _api.terminate()
        try:
            _api.wait(timeout=15)
        except subprocess.TimeoutExpired:
            _api.kill()
        _log.close()
        print("API đã tắt. Log:", _log_path)

## Bước 12. Đóng gói mô hình để tải về

Mô hình đã nằm trên Drive ở dạng thư mục (Bước 7.3). Ô này tạo thêm **một file `.zip`** cạnh thư mục đó, để tải về máy hoặc chép sang máy chủ: dùng cho Docker (`trien_khai/Dockerfile`) hoặc tải lên S3 cho SageMaker. Đóng gói mất khoảng 1–2 phút vì mô hình nặng khoảng 0,7 GB.

Nên tải file từ giao diện Google Drive thay vì `DOWNLOAD_NOW`, vì tải file lớn qua trình duyệt từ Colab dễ bị ngắt.

In [ ]:
#@title Đóng gói
MAKE_ZIP = True  #@param {type:"boolean"}
DOWNLOAD_NOW = False  #@param {type:"boolean"}
if MAKE_ZIP:
    _name = os.path.basename(MODEL_STORE) if os.path.abspath(TEST_MODEL) == os.path.abspath(FINAL_DIR) \
        else os.path.basename(os.path.normpath(TEST_MODEL))
    _zip_base = os.path.join(PROJECT_DIR, "models", _name)
    os.makedirs(os.path.dirname(_zip_base), exist_ok=True)
    MODEL_ZIP = shutil.make_archive(_zip_base, "zip", root_dir=os.path.dirname(os.path.normpath(TEST_MODEL)),
                                    base_dir=os.path.basename(os.path.normpath(TEST_MODEL)))
    print(f"✅ {MODEL_ZIP} ({os.path.getsize(MODEL_ZIP) / 1024 ** 3:.2f} GB)")
    if DOWNLOAD_NOW and IN_COLAB:
        from google.colab import files as _colab_files
        _colab_files.download(MODEL_ZIP)

## Tóm tắt

In [ ]:
#@title Tóm tắt lần chạy
_sum = {
    "run_id": RUN_ID, "data_source": DATA_SOURCE, "model_tested": TEST_MODEL,
    "model_saved": MODEL_STORE if os.path.isdir(MODEL_STORE) else None,
    "test_f1": globals().get("TEST_METRICS", {}).get("f1"),
    "run_dir": RUN_DIR,
}
with open(os.path.join(RUN_DIR, "tom_tat.json"), "w", encoding="utf-8") as f:
    json.dump(_sum, f, ensure_ascii=False, indent=2)
for k, v in _sum.items():
    print(f"{k:<14} {v}")
if "TEST_REPORT" in globals():
    display(TEST_REPORT)

## Bước tiếp theo

- **Dữ liệu thật:** gán nhãn theo `huong_dan_gan_nhan_ner_tien_hiep.docx` và `README.md` của bộ công cụ. Sau đó tải các file export lên `PROJECT_DIR/exports/`, đặt `DATA_SOURCE = label_studio`, rồi **Run all**.
- **So sánh các lần chạy:** mỗi lần chạy có một thư mục `runs/<RUN_ID>/` riêng. Hãy so `bao_cao/ket_qua_test.csv` giữa các lần, và luôn dùng **cùng một tập test**.
- **Triển khai:** lấy thư mục hoặc file zip trong `PROJECT_DIR/models/` rồi làm theo `trien_khai/README.md` (Docker, API, xử lý file hàng loạt).

## Xử lý sự cố trên Colab

| Vấn đề | Cách xử lý |
|---|---|
| Không có GPU, hoặc Colab báo đã hết hạn mức GPU | Bản Colab miễn phí giới hạn thời gian dùng GPU. Đợi một thời gian rồi thử lại, dùng gói trả phí, hoặc chạy trên CPU với `NUM_EPOCHS` nhỏ (rất chậm) |
| Mất kết nối khi đang huấn luyện | Bật `SAVE_CHECKPOINTS_TO_DRIVE` từ đầu. Khi chạy lại, bật `RESUME_FROM_DRIVE_CHECKPOINT` |
| `CUDA out of memory` | Giảm `BATCH_SIZE` xuống 8, hoặc `MAX_LEN` xuống 128 |
| Loss bằng `nan` | Giảm `LEARNING_RATE` xuống `2e-5` |
| Lỗi import sau khi cài lại thư viện | **Runtime → Restart session**, rồi chạy lại từ Bước 1 |
| Pip báo xung đột phiên bản | Bỏ qua được, nếu ô Bước 2 in ra được phiên bản `transformers` |
| Không thấy file zip bộ công cụ | Kiểm tra `PROJECT_DIR`, tên file `bo_cong_cu_ner_tien_hiep*.zip`, hoặc điền `TOOLKIT_ZIP` |
| Drive đầy | Mỗi checkpoint khoảng 2 GB, mỗi mô hình khoảng 0,7 GB (thêm 0,7 GB nếu có file zip). Xoá `checkpoints/` và các mô hình cũ không dùng |
| Phiên Colab khởi động lại, Bước 8 báo không thấy mô hình | Điền `MODEL_FOR_TEST` là thư mục mô hình trên Drive (`PROJECT_DIR/models/…`) |
| API không khởi động được | Xem `runs/<RUN_ID>/api_log.txt` |